# Model Validation and Performance Assessment

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Mock df_final for independent execution of validation notebook
# This should mirror the structure of df_final from the main notebook
df = pd.DataFrame({
    'person_income': [50000, 60000, 70000, 80000, 1000000, 55000, 65000, 75000, 85000, 95000],
    'loan_status': [0, 0, 1, 0, 1, 0, 1, 0, 1, 0],
    'person_home_ownership': ['RENT', 'MORTGAGE', 'RENT', 'MORTGAGE', 'OWN', 'RENT', 'MORTGAGE', 'RENT', 'OWN', 'RENT'],
    'cb_person_cred_hist_length': [3, 5, 2, 10, 40, 4, 6, 2, 12, 5]
})

# Imputation
num_imputer = SimpleImputer(strategy='median')
df['person_income'] = num_imputer.fit_transform(df[['person_income']]).ravel()

# Outlier Capping (99th percentile)
def cap_outliers(series, percentile=0.99):
    return series.clip(upper=series.quantile(percentile))

df['person_income'] = cap_outliers(df['person_income'])

# Encoding
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_cols = encoder.fit_transform(df[['person_home_ownership']])
encoded_df = pd.DataFrame(encoded_cols, columns=encoder.get_feature_names_out(['person_home_ownership']))
df_final = pd.concat([df.drop(columns=['person_home_ownership']), encoded_df], axis=1)

# Feature Engineering & Mock Targets
df_final['mock_recovery_rate'] = np.random.uniform(0.1, 0.9, size=len(df_final))
df_final['mock_exposure'] = df_final['person_income'] * np.random.uniform(0.1, 0.4, size=len(df_final))
df_final['loan_to_income_ratio'] = df_final['mock_exposure'] / df_final['person_income']

# Additional features used in the notebook for full compatibility
df_final['history_income_index'] = df_final['cb_person_cred_hist_length'] * np.log1p(df_final['person_income'])
min_income = df_final['person_income'].min()
max_income = df_final['person_income'].max()
df_final['recovery_potential_score'] = (df_final['person_income'] - min_income) / (max_income - min_income)


## Model Validation Strategy

To ensure our credit risk models (PD, LGD, EAD) are stable and not overfit to the training data, we follow these steps:
1. **Train-Test Split**: We split the data (70% Train / 30% Test).
2. **Out-of-Sample Testing**: We evaluate the models on the test set to check for consistent performance.
3. **Stability Analysis**: We compare training and testing metrics (like AUC or RMSE) to ensure the gap is minimal.

## Integrated Model Validation and Performance Assessment

This section combines data splitting, model training for the three Basel pillars, and comprehensive evaluation.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, mean_squared_error, r2_score
import numpy as np

# 1. Data Splitting
X = df_final.drop(columns=['loan_status', 'mock_recovery_rate', 'mock_exposure'])
y_pd = df_final['loan_status']

X_train, X_test, y_train_pd, y_test_pd = train_test_split(
    X, y_pd, test_size=0.3, random_state=42, stratify=y_pd
)

# 2. Model Training
# PD Model
pd_model = LogisticRegression(max_iter=1000).fit(X_train, y_train_pd)

# LGD Model (subset of defaults)
lgd_train_idx = X_train.index[y_train_pd == 1]
X_train_lgd = X_train.loc[lgd_train_idx]
y_train_lgd = df_final.loc[lgd_train_idx, 'mock_recovery_rate']
lgd_model = LinearRegression().fit(X_train_lgd, y_train_lgd)

# EAD Model
ead_model = LinearRegression().fit(X_train, df_final.loc[X_train.index, 'mock_exposure'])

# 3. Validation and Metrics
pd_probs = pd_model.predict_proba(X_test)[:, 1]
ead_preds = ead_model.predict(X_test)
lgd_preds = lgd_model.predict(X_test)

# Calculate metrics
auc = roc_auc_score(y_test_pd, pd_probs)
ead_rmse = np.sqrt(mean_squared_error(df_final.loc[X_test.index, 'mock_exposure'], ead_preds))

# 4. Final Expected Loss Estimation on Test Set
lgd_factor = np.clip(1 - lgd_preds, 0, 1)
el_estimates = pd_probs * lgd_factor * np.maximum(ead_preds, 0)

validation_summary = pd.DataFrame({
    'Actual_Status': y_test_pd,
    'PD_Probability': pd_probs,
    'LGD_Factor': lgd_factor,
    'EAD_Estimate': np.maximum(ead_preds, 0),
    'Expected_Loss': el_estimates
})

print(f"Validation Complete.")
print(f"PD AUC-ROC: {auc:.2f}")
print(f"EAD RMSE: {ead_rmse:.2f}")
display(validation_summary)
